In [1]:
import pandas as pd
import os
from collections import defaultdict
import re
from tqdm import tqdm
import numpy as np

In [3]:
BASE_DIR = f"/gpfs/commons/groups/knowles_lab/vmazeeva/BigBrain/Processed/Train"

## ChromBPNet

In [4]:
overlap_dict = defaultdict(dict)

cell_types = ['microglia', 'astrocyte', 'neuron', 'oligodendrocyte']

running_total = 0
running_overlap_total = 0

for chrom in range(1,23):
    annotation_dir = os.path.join(BASE_DIR, "annotations", f"chr{chrom}")
    
    for cell in cell_types:
        chrombp=pd.read_csv(f"/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/ADSP_vcf/58K_preview_compact/rare_variants/chrombpnet/variant_peak_pairs_scored/rare_variant_chrombpnet_{cell}_chr{chrom}_variant_peak_pairs_scored.csv", 
                            sep = "\t")
        print(chrombp.columns)
        print(chrombp.head)
        print(len(chrombp['BP']), len(set(chrombp['BP'].to_list())))
        break

        for file in sorted(os.listdir(annotation_dir)):
            if not file.endswith("_annotations.tsv.gz"):
                continue

            gene = re.sub(r"_annotations\.tsv\.gz$", "", file)
            anno_df = pd.read_csv(os.path.join(annotation_dir, file), sep="\t", usecols=['pos'])

            n_total = len(set(anno_df['pos'].to_list()))
            n_overlap = len((set(chrombp['BP'].to_list())) & (set(anno_df['pos'].to_list())))

            running_total += n_total
            running_overlap_total += n_overlap

            overlap_dict[chrom][gene] = dict()
            overlap_dict[chrom][gene][cell] = (n_total, n_overlap)
    break



Index(['Chr', 'PeakID', 'peak_center', 'SNP', 'CM', 'BP', 'ALT', 'REF',
       'distance_from_center', 'annotationSimple', 'assay',
       'log_counts_diff_chrombpnet', 'log_probs_diff_abs_sum_chrombpnet',
       'probs_jsd_diff_chrombpnet'],
      dtype='object')
<bound method NDFrame.head of          Chr                       PeakID  peak_center              SNP   CM  \
0          1                     Peak_258      3796547    1:3795538:C:G  0.0   
1          1                     Peak_258      3796547    1:3795550:T:C  0.0   
2          1                     Peak_258      3796547    1:3795575:A:T  0.0   
3          1                     Peak_258      3796547    1:3795589:G:A  0.0   
4          1                     Peak_258      3796547    1:3795615:A:G  0.0   
...      ...                          ...          ...              ...  ...   
1800514    1  microglia_H3K4me3_peak_4139    248906494  1:248907470:A:G  0.0   
1800515    1  microglia_H3K4me3_peak_4139    248906494  1:2489074

In [24]:
overlap_dict

defaultdict(dict,
            {1: {'ENSG00000000457': {'astrocyte': (5874, 205)},
              'ENSG00000000460': {'astrocyte': (10027, 340)},
              'ENSG00000000938': {'astrocyte': (5674, 488)},
              'ENSG00000000971': {'astrocyte': (6236, 59)},
              'ENSG00000001460': {'astrocyte': (6579, 351)},
              'ENSG00000001461': {'astrocyte': (6463, 354)},
              'ENSG00000004455': {'astrocyte': (6574, 434)},
              'ENSG00000004487': {'astrocyte': (6126, 357)},
              'ENSG00000006555': {'astrocyte': (6172, 266)},
              'ENSG00000007341': {'astrocyte': (7139, 682)},
              'ENSG00000007908': {'astrocyte': (6380, 180)},
              'ENSG00000007923': {'astrocyte': (7433, 885)},
              'ENSG00000007933': {'astrocyte': (5843, 166)},
              'ENSG00000007968': {'astrocyte': (5852, 474)},
              'ENSG00000008118': {'astrocyte': (6161, 473)},
              'ENSG00000008128': {'astrocyte': (6112, 0)},
     

## Enformer

In [14]:
def scale(df):
    df = np.arcsinh(df)
    return (df-df.min())/ (df.max() - df.min())

overlap_dict = defaultdict(dict)

cell_types = ['microglia', 'astrocyte', 'neuron', 'oligodendrocyte']

running_total = 0
running_overlap_total = 0

for chrom in range(1,23):
    annotation_dir = os.path.join(BASE_DIR, "annotations", f"chr{chrom}")
    
    for cell in cell_types:
        print(cell)
        path = f"/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/Alzheimer-RV/data/gene_matrices_maf/ADSP_rare_variants_enformer_delta_scores_annotations_{cell}.tsv.gz"
        newpath = f"/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/Alzheimer-RV/data/gene_matrices_maf/enformer_scored_06_03_2025/annotations_scaled/ADSP_rare_variants_06_03_2025_enformer_delta_scores_annotations_{cell}.tsv.gz"
        enformer1 = pd.read_csv(path, 
                            sep = "\t")
        print(enformer1.columns)
        enformer2 = pd.read_csv(newpath,
                            sep = "\t")
        print(enformer2.columns)
        # enformer2 = enformer2.rename(columns={'POS': 'pos', 'REF': 'ref', 'ALT': 'alt'})
        # enformer = pd.concat([enformer1, enformer2])
        # enformer = pd.concat([enformer1, enformer2])
        # print(enformer.columns)
        # print(enformer.head)
        # break
        # enformer = enformer[enformer['CHR']==int(chrom)]

        # for file in sorted(os.listdir(annotation_dir)):
        #     if not file.endswith("_annotations.tsv.gz"):
        #         continue

        #     gene = re.sub(r"_annotations\.tsv\.gz$", "", file)
        #     anno_df = pd.read_csv(os.path.join(annotation_dir, file), sep="\t")

        #     n_total = len(set(anno_df['pos'].astype(float).to_list()))
        #     n_overlap1 = len((set(enformer['BP'].to_list())) & (set(anno_df['pos'].to_list())))
        #     n_overlap2 = len((set(enformer2['POS'].to_list())) & (set(anno_df['pos'].to_list())))

        #     running_total += n_total
        #     running_overlap_total += n_overlap1 

        #     overlap_dict[chrom][gene] = dict()
        #     overlap_dict[chrom][gene][cell] = (n_total, n_overlap1, n_overlap2)
    break




microglia
Index(['CHR', 'SNP', 'BP', 'A1', 'A2', 'monocyte_DNASE_delta',
       'monocyte_H3K27ac_delta', 'monocyte_H3K27me3_delta',
       'monocyte_H3K4me3_delta', 'microglia_TF_delta_max',
       'microglia_TF_delta_min'],
      dtype='object')
Index(['CHR', 'SNP', 'POS', 'REF', 'ALT', 'monocyte_DNASE_delta',
       'monocyte_H3K27ac_delta', 'monocyte_H3K27me3_delta',
       'monocyte_H3K4me3_delta', 'microglia_TF_delta_max',
       'microglia_TF_delta_min'],
      dtype='object')
astrocyte
Index(['CHR', 'SNP', 'BP', 'A1', 'A2', 'astrocyte_TF_delta_max',
       'astrocyte_TF_delta_min'],
      dtype='object')
Index(['CHR', 'SNP', 'POS', 'REF', 'ALT', 'astrocyte_TF_delta_max',
       'astrocyte_TF_delta_min'],
      dtype='object')
neuron
Index(['CHR', 'SNP', 'BP', 'A1', 'A2', 'neuronal_TF_delta_max',
       'neuronal_TF_delta_min'],
      dtype='object')
Index(['CHR', 'SNP', 'POS', 'REF', 'ALT', 'neuronal_TF_delta_max',
       'neuronal_TF_delta_min'],
      dtype='object')
oligoden

In [32]:
overlap_dict

defaultdict(dict,
            {1: {'ENSG00000000457': {'microglia': (5874, 100, 341)},
              'ENSG00000000460': {'microglia': (10027, 208, 631)},
              'ENSG00000000938': {'microglia': (5674, 162, 1379)},
              'ENSG00000000971': {'microglia': (6236, 62, 0)},
              'ENSG00000001460': {'microglia': (6579, 141, 1610)},
              'ENSG00000001461': {'microglia': (6463, 134, 1784)},
              'ENSG00000004455': {'microglia': (6574, 155, 2008)},
              'ENSG00000004487': {'microglia': (6126, 82, 1407)},
              'ENSG00000006555': {'microglia': (6172, 212, 1448)},
              'ENSG00000007341': {'microglia': (7139, 256, 1749)},
              'ENSG00000007908': {'microglia': (6380, 115, 499)},
              'ENSG00000007923': {'microglia': (7433, 313, 2588)},
              'ENSG00000007933': {'microglia': (5843, 89, 155)},
              'ENSG00000007968': {'microglia': (5852, 152, 1471)},
              'ENSG00000008118': {'microglia': (61

## Alpha Missense

In [15]:
alpha_miss = '/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/annotations_hg38/merged_annotations_ADSP_v2/alphamissense/AlphaMissense_hg38.tsv.gz'
alpha = pd.read_csv(alpha_miss, sep = "\t", skiprows=3)
print(alpha.head)

<bound method NDFrame.head of          #CHROM       POS REF ALT genome uniprot_id             transcript_id  \
0          chr1     69094   G   T   hg38     Q8NH21         ENST00000335137.4   
1          chr1     69094   G   C   hg38     Q8NH21         ENST00000335137.4   
2          chr1     69094   G   A   hg38     Q8NH21         ENST00000335137.4   
3          chr1     69095   T   C   hg38     Q8NH21         ENST00000335137.4   
4          chr1     69095   T   A   hg38     Q8NH21         ENST00000335137.4   
...         ...       ...  ..  ..    ...        ...                       ...   
71697551   chrY  57196925   T   G   hg38     Q01113  ENST00000244174.10_PAR_Y   
71697552   chrY  57196925   T   C   hg38     Q01113  ENST00000244174.10_PAR_Y   
71697553   chrY  57196925   T   A   hg38     Q01113  ENST00000244174.10_PAR_Y   
71697554   chrY  57196926   C   G   hg38     Q01113  ENST00000244174.10_PAR_Y   
71697555   chrY  57196926   C   A   hg38     Q01113  ENST00000244174.10_PAR_Y  

In [16]:
alpha['#CHROM']

0           chr1
1           chr1
2           chr1
3           chr1
4           chr1
            ... 
71697551    chrY
71697552    chrY
71697553    chrY
71697554    chrY
71697555    chrY
Name: #CHROM, Length: 71697556, dtype: object

## LOF + Missense

In [3]:
CHRO_NB = 21
path = f"/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/ADSP_vcf/36K_QC/VEP_annotations/chr{CHRO_NB}_LOF.txt"
lof = pd.read_csv(path, sep = "\t", skiprows=4).reset_index().drop_duplicates("level_1")
path = f"/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/ADSP_vcf/36K_QC/VEP_annotations/chr{CHRO_NB}_missense.txt"
missense = pd.read_csv(path, sep = "\t", header = None).drop_duplicates(1)
# variants = np.array(data['pos'].astype(int))
# data['lof'] = np.where(pd.DataFrame(variants).isin(list(lof['level_1'].str.split(":").str[1].astype(int))), 1, 0)
# data['missense'] = np.where(pd.DataFrame(variants).isin(list(missense[1].str.split(":").str[1].astype(int))), 1, 0)


In [8]:
lof

,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,level_8,level_9,level_10,level_11,level_12,"## VEP command-line: vep --assembly GRCh38 --canonical --dir_plugins [PATH]/loftee --format vcf --input_file [PATH]/ADSP.chr21.vcf.gz --output_file [PATH]/chr21.txt --plugin LoF,[PATH]/loftee,[PATH]/human_ancestor.fa.gz --variant_class"
0,21_14510308_A/T,21:14510308,T,ENSG00000155307,ENST00000285670,Transcript,splice_donor_variant,-,-,-,-,-,-,IMPACT=HIGH;STRAND=-1;VARIANT_CLASS=SNV;LoF=HC...
4,rs1357059260,21:14512443,T,ENSG00000155307,ENST00000285670,Transcript,splice_donor_variant,-,-,-,-,-,-,IMPACT=HIGH;STRAND=-1;VARIANT_CLASS=SNV;LoF=HC...
8,rs757290848,21:14521221,A,ENSG00000155307,ENST00000285670,Transcript,"stop_gained,splice_region_variant",437,262,88,R/*,Cga/Tga,-,IMPACT=HIGH;STRAND=-1;VARIANT_CLASS=SNV;LoF=HC...
11,rs190362907,21:14582137,T,ENSG00000155307,ENST00000285670,Transcript,"stop_gained,splice_region_variant",435,260,87,W/*,tGg/tAg,-,IMPACT=HIGH;STRAND=-1;VARIANT_CLASS=SNV;LoF=HC...
13,rs985716497,21:14582202,T,ENSG00000155307,ENST00000285670,Transcript,stop_gained,370,195,65,C/*,tgC/tgA,-,IMPACT=HIGH;STRAND=-1;VARIANT_CLASS=SNV;LoF=HC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2027,rs777561357,21:46563919,G,ENSG00000160305,ENST00000400274,Transcript,stop_gained,4372,4139,1380,S/*,tCa/tGa,-,IMPACT=HIGH;STRAND=1;VARIANT_CLASS=SNV;LoF=HC;...
2030,rs766516434,21:46599504,T,ENSG00000160307,ENST00000291700,Transcript,splice_acceptor_variant,-,-,-,-,-,-,IMPACT=HIGH;STRAND=-1;VARIANT_CLASS=SNV;CANONI...
2033,rs1286619153,21:46644453,T,ENSG00000160310,ENST00000291705,Transcript,stop_gained,470,292,98,Q/*,Cag/Tag,-,IMPACT=HIGH;STRAND=1;VARIANT_CLASS=SNV;LoF=HC;...
2043,rs1260399381,21:46652872,T,ENSG00000160310,ENST00000455177,Transcript,stop_gained,501,502,168,E/*,Gag/Tag,-,"IMPACT=HIGH;STRAND=1;FLAGS=cds_start_NF,cds_en..."


In [9]:
missense

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,rs1982880189,21:14373621,C,ENSG00000155304,ENST00000285667,Transcript,missense_variant,1439,1412,471,N/S,aAc/aGc,-,IMPACT=MODERATE;STRAND=-1;VARIANT_CLASS=SNV;CA...
1,rs547143733,21:14373671,A,ENSG00000155304,ENST00000285667,Transcript,missense_variant,1389,1362,454,Q/H,caA/caT,-,IMPACT=MODERATE;STRAND=-1;VARIANT_CLASS=SNV;CA...
2,rs769171893,21:14373747,T,ENSG00000155304,ENST00000285667,Transcript,missense_variant,1313,1286,429,S/Y,tCt/tAt,-,IMPACT=MODERATE;STRAND=-1;VARIANT_CLASS=SNV;CA...
3,rs369475727,21:14373778,C,ENSG00000155304,ENST00000285667,Transcript,missense_variant,1282,1255,419,Q/E,Caa/Gaa,-,IMPACT=MODERATE;STRAND=-1;VARIANT_CLASS=SNV;CA...
4,rs200646454,21:14373790,A,ENSG00000155304,ENST00000285667,Transcript,missense_variant,1270,1243,415,R/C,Cgt/Tgt,-,IMPACT=MODERATE;STRAND=-1;VARIANT_CLASS=SNV;CA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65675,rs140169346,21:46663507,A,ENSG00000160310,ENST00000355680,Transcript,missense_variant,1477,1222,408,A/T,Gct/Act,-,IMPACT=MODERATE;STRAND=1;VARIANT_CLASS=SNV;CAN...
65680,rs1445304119,21:46663544,T,ENSG00000160310,ENST00000355680,Transcript,missense_variant,1514,1259,420,T/I,aCa/aTa,-,IMPACT=MODERATE;STRAND=1;VARIANT_CLASS=SNV;CAN...
65684,rs369158228,21:46664358,T,ENSG00000160310,ENST00000458387,Transcript,missense_variant,1115,890,297,A/V,gCa/gTa,-,IMPACT=MODERATE;STRAND=1;VARIANT_CLASS=SNV
65685,rs765598184,21:46664360,C,ENSG00000160310,ENST00000458387,Transcript,missense_variant,1117,892,298,Y/H,Tat/Cat,-,IMPACT=MODERATE;STRAND=1;VARIANT_CLASS=SNV


In [2]:
CHRO_NB = 21
abc = pd.DataFrame()
for cell in ['microglia','neuron','oligodendrocyte','astrocyte']:
    path_hg38new = f"/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/annotations/brain_all/glass_lab_fastq/processed_files/ABC_data/ABC_results_{cell}_v2/{cell}/Predictions/EnhancerPredictionsAllPutative.tsv.gz"
    cell_df= pd.read_csv(path_hg38new, sep = "\t")
    cell_df = cell_df[cell_df['ABC.Score']>=0.02]
    cell_df = cell_df[cell_df['chr']==f'chr{CHRO_NB}']
    cell_df['Cell'] = cell
    abc = pd.concat((abc, cell_df))
print(abc.head)


<bound method NDFrame.head of            chr     start       end                                name  \
5256379  chr21  14215841  14216761    promoter|chr21:14215841-14216761   
5256390  chr21  14216965  14217465       genic|chr21:14216965-14217465   
5256478  chr21  14272654  14274278  intergenic|chr21:14272654-14274278   
5256621  chr21  14382938  14383539    promoter|chr21:14382938-14383539   
5256622  chr21  14382938  14383539    promoter|chr21:14382938-14383539   
...        ...       ...       ...                                 ...   
4974806  chr21  46634923  46636405    promoter|chr21:46634923-46636405   
4974807  chr21  46634923  46636405    promoter|chr21:46634923-46636405   
4974808  chr21  46634923  46636405    promoter|chr21:46634923-46636405   
4974809  chr21  46634923  46636405    promoter|chr21:46634923-46636405   
4974810  chr21  46634923  46636405    promoter|chr21:46634923-46636405   

              class  activity_base  activity_base_enh  \
5256379    promoter     

In [3]:
abc.columns

Index(['chr', 'start', 'end', 'name', 'class', 'activity_base',
       'activity_base_enh', 'activity_base_squared_enh', 'normalized_atac_enh',
       'normalized_h3k27ac_enh', 'TargetGene', 'TargetGeneTSS',
       'TargetGeneExpression', 'TargetGenePromoterActivityQuantile',
       'TargetGeneIsExpressed', 'TargetGeneEnsembl_ID', 'normalized_atac_prom',
       'normalized_h3k27ac_prom', 'distance', 'isSelfPromoter',
       'powerlaw_contact', 'powerlaw_contact_reference', 'hic_contact',
       'hic_contact_pl_scaled', 'hic_pseudocount', 'hic_contact_pl_scaled_adj',
       'ABC.Score.Numerator', 'ABC.Score', 'powerlaw.Score.Numerator',
       'powerlaw.Score', 'CellType', 'hic_contact_squared', 'Cell'],
      dtype='object')

In [5]:
abc['TargetGeneEnsembl_ID']

5256379    ENSG00000185272
5256390    ENSG00000185272
5256478    ENSG00000185272
5256621    ENSG00000185272
5256622    ENSG00000155304
                ...       
4974806    ENSG00000160298
4974807    ENSG00000160299
4974808    ENSG00000160305
4974809    ENSG00000160307
4974810    ENSG00000160310
Name: TargetGeneEnsembl_ID, Length: 3217, dtype: object

In [33]:
import pandas as pd
import os

genotype_dir = "/gpfs/commons/groups/knowles_lab/vmazeeva/BigBrain/Processed/genotypes"
annotation_dir = "/gpfs/commons/groups/knowles_lab/vmazeeva/BigBrain/Processed/annotations"


ann_df = pd.read_csv(os.path.join(annotation_dir, "chr22", "ENSG00000100294_annotations.tsv.gz"), sep = "\t")
ann_df.head()



geno_df = pd.read_csv(os.path.join(genotype_dir, "chr22", "ENSG00000100294_genotypes.tsv.gz"), sep = "\t", index_col = "IID")
geno_df.head()






,22:43033017_C_A,22:43033027_T_C,22:43033220_T_C,22:43033334_C_T,22:43033930_G_A,22:43034113_A_G,22:43034131_T_C,22:43034173_T_G,22:43034335_A_G,22:43035440_A_G,...,22:43241699_C_G,22:43241758_T_C,22:43241863_T_C,22:43241973_A_T,22:43242477_T_C,22:43242646_G_T,22:43242729_A_G,22:43242811_C_G,22:43243132_G_A,22:43243260_C_G
IID,,,,,,,,,,,,,,,,,,,,,
CASE-NEUAE810KF7-9607-T,0.00142,0.0,0.002455,0.002457,0.002194,0.0,0.0,0.001291,0.0,0.001162,...,0.0,0,0,0,1,1.0,1,0.0,0,0
CASE-NEUAK233YVY-9803-T,0.00142,0.0,0.002455,0.002457,0.002194,0.0,0.0,0.001291,0.0,0.001162,...,0.0,0,0,0,0,0.0,0,0.0,0,0
CASE-NEUAL614GYB-8618-T,0.00142,0.0,0.002455,0.002457,0.002194,0.0,0.0,0.001291,0.0,0.001162,...,0.0,0,0,0,0,0.0,0,0.0,0,0
CASE-NEUAN180JPP-10533-T,0.00142,0.0,0.002455,0.002457,0.002194,0.0,0.0,0.001291,0.0,0.001162,...,0.0,0,0,0,0,0.0,0,0.0,0,0
CASE-NEUAP460GUX-10708-T,0.00142,0.0,0.002455,0.002457,0.002194,0.0,0.0,0.001291,0.0,0.001162,...,0.0,0,0,0,0,0.0,0,0.0,0,0


In [34]:
ann_df.head()


,variant_id,chr,pos,MAP20,phyloP17way_primate,phyloP30way_mammalian,phastCons17way_primate_rankscore,phastCons30way_mammalian,GERP_RS,bStatistic,...,log_counts_diff_chrombpnet_astrocyte,log_counts_diff_chrombpnet_neuron,log_counts_diff_chrombpnet_oligodendrocyte,alphamissense,splice,ABC_microglia,ABC_neuron,ABC_oligodendrocyte,ABC_astrocyte,dist_to_TSS
0,0,22.0,43033017.0,0.037500,0.276,0.319,0.48053,0.060,0.113,735.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.042879
1,1,22.0,43033027.0,0.000000,0.225,0.260,0.49222,0.057,0.113,735.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.042839
2,2,22.0,43033220.0,0.787500,-0.828,-0.727,0.08282,0.001,-0.226,735.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.042079
3,3,22.0,43033334.0,0.000000,0.249,0.275,0.71733,0.205,0.113,735.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.041630
4,4,22.0,43033930.0,0.616667,-0.096,0.716,0.49222,0.152,-1.650,735.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.039271


In [30]:
from collections import defaultdict
import os
import pandas as pd
import re

## add index column back to annotation matrices
ref_dir = "/gpfs/commons/groups/knowles_lab/vmazeeva/BigBrain/Processed/reference"
for chrom in range(1,23):
    if chrom == 21: continue
    for file in os.listdir(os.path.join(annotation_dir, f"chr{chrom}")):
        if not file.endswith("_annotations.tsv.gz"):
            continue

        gene = re.sub(r"_annotations\.tsv\.gz$", "", file)
        
        anno_df = pd.read_csv(os.path.join(annotation_dir, f"chr{chrom}", file), sep = "\t")
        geno_df = pd.read_csv(os.path.join(genotype_dir, f"chr{chrom}", f"{gene}_genotypes.tsv.gz"), sep = "\t", index_col = "IID")
        variant_ids = geno_df.columns
        pos_to_alleles = dict()
        for variant_id in variant_ids:
            chr, pos_alleles = variant_id.split(":") 
            pos, counted, other = pos_alleles.split("_")
            pos_to_alleles[pos] = (counted, other)

        ref_table = pd.read_csv(os.path.join(ref_dir, f"chr{chrom}_ref.tsv.gz"), sep = "\t",
                    compression="gzip",
                    names=["chr", "pos", "ref"],
                    dtype={"chr": str, "pos": int, "ref": str})
        anno_df2 = pd.merge(anno_df, ref_table, on = ['pos'], how = 'left')
        anno_varids = []
        for _, row in anno_df2.iterrows():
            pos = row['pos']
            ref = row['ref']
            alt = ''
            counted_other = pos_to_alleles[str(int(pos))]
            if ref == counted_other[0]:
                alt = counted_other[1]
            else:
                alt = counted_other[0]
            anno_varids.append(f"{int(row['chr_x'])}:{row['pos']}_{ref}_{alt}")
        assert len(anno_varids) == len(anno_df), f"Mismatch: {len(anno_varids)} genotype variants vs {len(anno_df)} annotations"
        assert len(geno_df.columns) == len(anno_varids), f"Mismatch: {len(geno_df.columns)} genotype variants vs {len(anno_varids)} annotations"
        anno_df2['variant_id'] = anno_varids
        anno_df2.drop(columns=['ref', 'chr_y'], inplace=True)
        anno_df2.rename(columns={'chr_x': 'chr'}, inplace=True)
        anno_df2.set_index('variant_id', inplace=True)
        anno_df.index.name = 'variant_id'
        assert anno_df.values.shape == anno_df2.values.shape, f"Mismatch: {anno_df.values.shape} annotation matrix vs {anno_df2.values.shape} annotation matrix"
        assert (anno_df.values == anno_df2.values).all(), "Annotation matrices do not match"
        anno_df.to_csv(os.path.join(annotation_dir, f"chr{chrom}", file), sep = "\t", index=True, compression="gzip")


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [27]:
anno_df2.head()

,chr,pos,MAP20,phyloP17way_primate,phyloP30way_mammalian,phastCons17way_primate_rankscore,phastCons30way_mammalian,GERP_RS,bStatistic,integrated_fitCons_score,...,log_counts_diff_chrombpnet_astrocyte,log_counts_diff_chrombpnet_neuron,log_counts_diff_chrombpnet_oligodendrocyte,alphamissense,splice,ABC_microglia,ABC_neuron,ABC_oligodendrocyte,ABC_astrocyte,dist_to_TSS
variant_id,,,,,,,,,,,,,,,,,,,,,
1:179740033.0_A_T,1.0,179740033.0,0.90,-0.415,0.431,0.14158,0.007,-0.352,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.139155
1:179740457.0_G_A,1.0,179740457.0,1.00,-0.487,0.009,0.18454,0.003,0.462,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137816
1:179740573.0_C_G,1.0,179740573.0,1.00,-0.390,-0.048,0.77633,0.000,-2.040,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137449
1:179740627.0_C_G,1.0,179740627.0,1.00,-0.649,-0.533,0.39275,0.004,-0.708,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137278
1:179740695.0_G_A,1.0,179740695.0,0.95,0.498,0.003,0.71945,0.010,-0.249,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137063


In [20]:
geno_df.head()

,1:179740033_T_A,1:179740457_G_A,1:179740573_G_C,1:179740627_G_C,1:179740695_A_G,1:179740726_T_G,1:179740913_G_A,1:179740958_C_T,1:179740990_T_C,1:179741218_T_C,...,1:179975486_T_G,1:179975711_C_G,1:179975766_G_A,1:179975793_A_C,1:179976102_A_G,1:179976255_G_A,1:179976886_T_G,1:179977141_T_G,1:179977171_C_G,1:179977711_G_A
IID,,,,,,,,,,,,,,,,,,,,,
CASE-NEUAE810KF7-9607-T,0.0,2.0,0,0.0,0,0,0,0.0,0.000903,0,...,0.0,0.0,1,1.0,0.0,1.0,0,0,0,0.0
CASE-NEUAK233YVY-9803-T,0.0,1.0,0,0.0,0,0,0,0.0,0.000903,0,...,0.0,0.0,0,0.0,0.0,2.0,0,0,0,0.0
CASE-NEUAL614GYB-8618-T,1.0,0.0,0,1.0,0,0,0,0.0,0.000903,0,...,0.0,0.0,1,1.0,0.0,0.0,0,0,0,0.0
CASE-NEUAN180JPP-10533-T,1.0,1.0,0,1.0,0,0,0,0.0,0.000903,0,...,0.0,0.0,0,1.0,0.0,0.0,0,1,0,0.0
CASE-NEUAP460GUX-10708-T,1.0,1.0,0,1.0,0,0,0,0.0,0.000903,0,...,0.0,0.0,1,1.0,0.0,0.0,0,0,0,0.0


In [19]:
anno_df.head()

,chr,pos,MAP20,phyloP17way_primate,phyloP30way_mammalian,phastCons17way_primate_rankscore,phastCons30way_mammalian,GERP_RS,bStatistic,integrated_fitCons_score,...,log_counts_diff_chrombpnet_astrocyte,log_counts_diff_chrombpnet_neuron,log_counts_diff_chrombpnet_oligodendrocyte,alphamissense,splice,ABC_microglia,ABC_neuron,ABC_oligodendrocyte,ABC_astrocyte,dist_to_TSS
0,1.0,179740033.0,0.90,-0.415,0.431,0.14158,0.007,-0.352,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.139155
1,1.0,179740457.0,1.00,-0.487,0.009,0.18454,0.003,0.462,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137816
2,1.0,179740573.0,1.00,-0.390,-0.048,0.77633,0.000,-2.040,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137449
3,1.0,179740627.0,1.00,-0.649,-0.533,0.39275,0.004,-0.708,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137278
4,1.0,179740695.0,0.95,0.498,0.003,0.71945,0.010,-0.249,764.0,0.06567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.137063


In [10]:
pos_to_alleles

defaultdict(list,
            {'179740033': ('T', 'A'),
             '179740457': ('G', 'A'),
             '179740573': ('G', 'C'),
             '179740627': ('G', 'C'),
             '179740695': ('A', 'G'),
             '179740726': ('T', 'G'),
             '179740913': ('G', 'A'),
             '179740958': ('C', 'T'),
             '179740990': ('T', 'C'),
             '179741218': ('T', 'C'),
             '179741285': ('C', 'T'),
             '179741524': ('T', 'C'),
             '179741531': ('G', 'A'),
             '179741933': ('C', 'T'),
             '179742503': ('T', 'A'),
             '179742514': ('G', 'T'),
             '179742690': ('C', 'A'),
             '179742751': ('A', 'C'),
             '179742755': ('A', 'C'),
             '179743039': ('T', 'C'),
             '179743088': ('C', 'A'),
             '179743176': ('A', 'C'),
             '179743185': ('T', 'G'),
             '179743263': ('A', 'C'),
             '179743304': ('T', 'C'),
             '179743320': ('A', 